# 02 — Baseline Models: Regression Family
Linear, Ridge, Lasso, ElasticNet, PLS — with KFold and LOGO cross-validation.

**Run after:** `01_eda_correlation.ipynb` (uses `data/processed/processed.csv`)  
Outputs → `artifacts/`


In [1]:
import warnings, os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, ElasticNetCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42); random.seed(42)
sns.set_style("whitegrid")
os.makedirs("artifacts", exist_ok=True)

TARGETS  = ["Leaf_TPC","Root_TPC","Leaf_TFC","Root_TFC"]
FEATURES = [
    "Temp","Humid","CO2ppm","TChl","Car",
    "Dio-RC","Eto-RC","Fv-Fm",
    "Leaf_ExtractionYield","Root_ExtractionYield",
    "month_sin","month_cos"
]
print("Ready.")

Ready.


## 1. Load Preprocessed Data

In [2]:
# 01_preprocessing 또는 아래 인라인 전처리 실행
PROC_P = "data/processed/processed.csv"
RAW_P  = "rawdata.csv"

def scenario_label(co2):
    if co2 < 633:   return "SSP1-2.6"
    elif co2 < 961: return "SSP3-7.0"
    else:           return "SSP5-8.5"

def ensure_month_feats(df):
    if "month_sin" not in df.columns and "month" in df.columns:
        m = df["month"].astype(float)
        df["month_sin"] = np.sin(2*np.pi*m/12)
        df["month_cos"] = np.cos(2*np.pi*m/12)
    return df

if os.path.exists(PROC_P):
    df = pd.read_csv(PROC_P)
else:
    # 전처리 없이 rawdata에서 직접 스케일링
    df = pd.read_csv(RAW_P)
    df = ensure_month_feats(df)
    if "scenario_group" not in df.columns:
        df["scenario_group"] = df["CO2ppm"].apply(scenario_label)
    num_feats = [f for f in FEATURES if f not in ["month_sin","month_cos"]]
    scaler = RobustScaler()
    df[num_feats] = scaler.fit_transform(df[num_feats])

df = ensure_month_feats(df)
if "scenario_group" not in df.columns:
    df["scenario_group"] = df["CO2ppm"].apply(scenario_label)

X = df[FEATURES].values
y = df[TARGETS].values
groups = df["scenario_group"].values

print(f"X: {X.shape}, y: {y.shape}")
print("Scenario counts:", pd.Series(groups).value_counts().to_dict())

X: (405, 12), y: (405, 4)
Scenario counts: {'SSP5-8.5': 138, 'SSP1-2.6': 135, 'SSP3-7.0': 132}


## 2. Cross-Validation Helpers

In [ ]:
def metrics_per_target(y_true, y_pred, target_names):
    rows = []
    for i, tgt in enumerate(target_names):
        mae  = mean_absolute_error(y_true[:,i], y_pred[:,i])
        rmse = np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i]))
        r2   = r2_score(y_true[:,i], y_pred[:,i])
        rows.append({"target": tgt, "MAE": mae, "RMSE": rmse, "R2": r2})
    df_m = pd.DataFrame(rows)
    df_m.loc["mean"] = ["MEAN",
                        df_m["MAE"].mean(), df_m["RMSE"].mean(), df_m["R2"].mean()]
    return df_m

def run_kfold(model_fn, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    preds = np.zeros_like(y, dtype=float)
    for tr, te in kf.split(X):
        m = model_fn()
        m.fit(X[tr], y[tr])
        preds[te] = m.predict(X[te])
    return metrics_per_target(y, preds, TARGETS)

def run_logo(model_fn, X, y, groups):
    logo = LeaveOneGroupOut()
    preds = np.zeros_like(y, dtype=float)
    for tr, te in logo.split(X, y, groups):
        m = model_fn()
        m.fit(X[tr], y[tr])
        preds[te] = m.predict(X[te])
    return metrics_per_target(y, preds, TARGETS)

print("Helpers ready.")

Helpers ready.


## 3. Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

print("[Linear Regression — KFold]")
kf_linear = run_kfold(LinearRegression, X, y)
print(kf_linear.to_string(index=False))

print("\n[Linear Regression — LOGO]")
logo_linear = run_logo(LinearRegression, X, y, groups)
print(logo_linear.to_string(index=False))

[Linear Regression — KFold]
  target      MAE     RMSE       R2
Leaf_TPC 0.220262 0.282892 0.687361
Root_TPC 0.198310 0.272334 0.753877
Leaf_TFC 0.569230 0.717343 0.922214
Root_TFC 0.040948 0.056573 0.827469
    MEAN 0.257188 0.332285 0.797730

[Linear Regression — LOGO]
  target      MAE     RMSE       R2
Leaf_TPC 0.402398 0.501691 0.016725
Root_TPC 0.382545 0.507603 0.144938
Leaf_TFC 0.889142 1.190892 0.785616
Root_TFC 0.085843 0.107653 0.375251
    MEAN 0.439982 0.576960 0.330633


## 4. Ridge

In [ ]:
print("[Ridge — KFold]")
kf_ridge = run_kfold(lambda: Ridge(alpha=1.0, random_state=42), X, y)
print(kf_ridge.to_string(index=False))

print("\n[Ridge — LOGO]")
logo_ridge = run_logo(lambda: Ridge(alpha=1.0, random_state=42), X, y, groups)
print(logo_ridge.to_string(index=False))

[Ridge — KFold]
  target      MAE     RMSE       R2
Leaf_TPC 0.221428 0.285940 0.680587
Root_TPC 0.197821 0.272129 0.754246
Leaf_TFC 0.562512 0.718387 0.921988
Root_TFC 0.040893 0.056809 0.826027
    MEAN 0.255664 0.333316 0.795712

[Ridge — LOGO]
  target      MAE     RMSE        R2
Leaf_TPC 0.409524 0.515063 -0.036388
Root_TPC 0.379789 0.500359  0.169169
Leaf_TFC 0.858461 1.124273  0.808931
Root_TFC 0.084909 0.107003  0.382773
    MEAN 0.433171 0.561674  0.331121


## 5. Lasso & ElasticNet

In [ ]:
from sklearn.linear_model import Lasso, MultiTaskElasticNetCV

print("[Lasso — KFold-5]")
kf_lasso = run_kfold(
    lambda: Lasso(alpha=0.001, max_iter=10000, random_state=42),
    X, y
)
print(kf_lasso.to_string(index=False))

# ── Lasso LOGO: KFold와 공정 비교를 위해 추가 ──
print("\n[Lasso — LOGO (공정 비교)]")
logo_lasso = run_logo(
    lambda: Lasso(alpha=0.001, max_iter=10000, random_state=42),
    X, y, groups
)
print(logo_lasso.to_string(index=False))

print("\n[ElasticNet — LOGO]")
logo_enet = run_logo(
    lambda: MultiTaskElasticNetCV(l1_ratio=0.5, max_iter=10000, cv=3, random_state=42),
    X, y, groups
)
print(logo_enet.to_string(index=False))


## 6. PLS Regression

In [ ]:
def pls_pred(X_tr, y_tr, X_te, n=5):
    m = PLSRegression(n_components=n)
    m.fit(X_tr, y_tr)
    return m.predict(X_te)

# LOGO
logo = LeaveOneGroupOut()
preds_pls = np.zeros_like(y, dtype=float)
for tr, te in logo.split(X, y, groups):
    preds_pls[te] = pls_pred(X[tr], y[tr], X[te], n=5)
logo_pls = metrics_per_target(y, preds_pls, TARGETS)
print("[PLS n=5 — LOGO]")
print(logo_pls.to_string(index=False))

[PLS n=5 — LOGO]
  target      MAE     RMSE        R2
Leaf_TPC 0.427112 0.526041 -0.081039
Root_TPC 0.368601 0.467022  0.276189
Leaf_TFC 1.230679 1.518789  0.651308
Root_TFC 0.089017 0.109187  0.357319
    MEAN 0.528852 0.655260  0.300944


## 7. Summary: Regression Family Comparison (LOGO 통일)

모든 모델을 동일한 LOGO 기준으로 비교합니다.  
Lasso는 위에서 KFold / LOGO 두 방식 모두 실행 — LOGO 수치를 사용합니다.


In [ ]:
summary = pd.DataFrame({
    "Model":    ["Linear","Ridge","Lasso","ElasticNet","PLS"],
    "CV":       ["LOGO","LOGO","LOGO","LOGO","LOGO"],
    "MAE_mean": [
        float(logo_linear.loc[logo_linear["target"]=="MEAN","MAE"]),
        float(logo_ridge.loc[logo_ridge["target"]=="MEAN","MAE"]),
        float(logo_lasso.loc[logo_lasso["target"]=="MEAN","MAE"]),
        float(logo_enet.loc[logo_enet["target"]=="MEAN","MAE"]),
        float(logo_pls.loc[logo_pls["target"]=="MEAN","MAE"]),
    ],
    "R2_mean": [
        float(logo_linear.loc[logo_linear["target"]=="MEAN","R2"]),
        float(logo_ridge.loc[logo_ridge["target"]=="MEAN","R2"]),
        float(logo_lasso.loc[logo_lasso["target"]=="MEAN","R2"]),
        float(logo_enet.loc[logo_enet["target"]=="MEAN","R2"]),
        float(logo_pls.loc[logo_pls["target"]=="MEAN","R2"]),
    ],
}).sort_values("MAE_mean")

print("=== Regression Family: LOGO 통일 비교 ===")
print(summary.to_string(index=False))
summary.to_csv("artifacts/baseline_regression_summary.csv", index=False)
print("\nSaved → artifacts/baseline_regression_summary.csv")
